<a href="https://colab.research.google.com/github/Muen1/multilingual-health-qa-africa/blob/main/notebook/03_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# CELL 0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os

BASE_DIR = '/content/drive/MyDrive/multilingual_health_qa'
DATA_DIR = f'{BASE_DIR}/data'
OUT_DIR  = f'{BASE_DIR}/outputs'
PLOT_DIR = f'{BASE_DIR}/plots'

for d in [OUT_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted.")
print("Data files:", os.listdir(DATA_DIR))

Mounted at /content/drive
Drive mounted.
Data files: ['Train.csv', 'Val.csv', 'Test.csv', 'SampleSubmission.csv', 'train_clean.csv', 'val_clean.csv', 'test_clean.csv', 'predictions_exp01_mt5small_zeroshot.csv']


In [4]:
# CELL S2 — Install all dependencies
!pip install -q --upgrade pip

!pip install -q \
    transformers==4.44.0 \
    datasets==2.19.0 \
    peft==0.10.0 \
    accelerate==0.29.3 \
    bitsandbytes==0.43.1 \
    rouge-score==0.1.2 \
    sentencepiece==0.2.0 \
    protobuf==3.20.3 \
    evaluate==0.4.1 \
    fsspec==2025.3.0

# Verify no critical conflicts remain
import pkg_resources
packages = ['transformers', 'datasets', 'peft', 'accelerate', 'evaluate']
print("\n=== Installed versions ===")
for p in packages:
    try:
        print(f"  {p}: {pkg_resources.get_distribution(p).version}")
    except:
        print(f"  {p}: NOT FOUND")

print("\nInstallation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 6.5 MB/s eta 0:00:00
ERROR: Cannot install datasets==2.19.0 and fsspec==2025.3.0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts

=== Installed versions ===
  transformers: 4.40.0
  datasets: 2.19.0
  peft: 0.10.0
  accelerate: 0.29.3
  evaluate: 0.4.6

Installation complete.


/tmp/ipykernel_973/165690623.py:7: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [5]:
# CELL S2 — REPLACE with this. Do NOT reinstall anything.
# Your Colab already has compatible versions pre-installed.
# The "errors" you saw are just warnings about optional packages — ignore them.

import importlib

required = {
    'transformers': '4.40.0',
    'datasets':     '2.19.0',
    'peft':         '0.10.0',
    'accelerate':   '0.29.3',
    'evaluate':     '0.4.6',
    'rouge_score':  None,
    'sentencepiece':None,
}

print("=== Package check ===")
all_ok = True
for package, expected in required.items():
    try:
        mod = importlib.import_module(package)
        version = getattr(mod, '__version__', 'installed')
        print(f"  ✓ {package}: {version}")
    except ImportError:
        print(f"  ✗ {package}: MISSING")
        all_ok = False

if all_ok:
    print("\nAll packages present. Proceed to Cell S3.")
else:
    print("\nSome packages missing — run: !pip install -q peft evaluate rouge-score sentencepiece")

=== Package check ===
  ✓ transformers: 4.40.0
  ✓ datasets: 2.19.0
  ✓ peft: 0.10.0
  ✓ accelerate: 0.29.3
  ✓ evaluate: 0.4.6
  ✓ rouge_score: installed
  ✓ sentencepiece: 0.2.1

All packages present. Proceed to Cell S3.


In [6]:
# imports
import gc
import os
import json
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import evaluate as hf_evaluate
from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : Tesla T4


In [8]:
# CELL S4 — Load cleaned data
train = pd.read_csv(f'{DATA_DIR}/train_clean.csv')
val   = pd.read_csv(f'{DATA_DIR}/val_clean.csv')
test  = pd.read_csv(f'{DATA_DIR}/test_clean.csv')

# Column mapping for this dataset
q_col    = 'input'
a_col    = 'output'
lang_col = 'subset'
id_col   = 'ID'

# Rebuild prompts v1 if missing
if 'input_text' not in train.columns:
    def _prompt_v1(q, l):
        return f"Language: {l}\nQuestion: {q}\nAnswer:"
    train['input_text']  = train.apply(lambda r: _prompt_v1(r[q_col], r[lang_col]), axis=1)
    train['target_text'] = train[a_col].astype(str)
    val['input_text']    = val.apply(lambda r: _prompt_v1(r[q_col], r[lang_col]), axis=1)
    val['target_text']   = val[a_col].astype(str)
    test['input_text']   = test.apply(lambda r: _prompt_v1(r[q_col], r[lang_col]), axis=1)

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")
print("Sample input:", train['input_text'].iloc[0])
print("Sample target:", train['target_text'].iloc[0][:150])

Train: 29814, Val: 6686, Test: 2618
Sample input: Language: Aka_Gha
Question: Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye wɔn werɛ, sɛnea wɔbɛboa wɔn ma wɔanya mmoa firi nnwumakuo a ɛfata hɔ, ne sɛnea wɔbɛsiw afɔbu suban ne nsɛm a nkurɔfoɔ de gu oyarefoɔ no so no kwan.
Answer:
Sample target: Mmabun betumi aboa atipɛnfo a ebia nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ so denam: Nkate fam mmoa a wɔde bɛma na wɔagye wɔn nkate atom a wɔremmu atɛ


In [10]:
# CELL S5 — Shared helper functions (used by all experiments)

rouge_metric = hf_evaluate.load("rouge")

def get_tokenized_datasets(tokenizer, train_df, val_df,
                            max_input=256, max_target=256):
    """Tokenize train and val DataFrames into HuggingFace Datasets."""
    def preprocess(examples):
        inputs = tokenizer(
            examples['input_text'],
            max_length=max_input,
            truncation=True,
            padding='max_length'
        )
        targets = tokenizer(
            examples['target_text'],
            max_length=max_target,
            truncation=True,
            padding='max_length'
        )
        labels = [
            [(l if l != tokenizer.pad_token_id else -100) for l in label]
            for label in targets['input_ids']
        ]
        inputs['labels'] = labels
        return inputs

    train_ds = Dataset.from_pandas(
        train_df[['input_text','target_text']].reset_index(drop=True))
    val_ds   = Dataset.from_pandas(
        val_df[['input_text','target_text']].reset_index(drop=True))

    train_tok = train_ds.map(preprocess, batched=True,
                             remove_columns=['input_text','target_text'])
    val_tok   = val_ds.map(preprocess,   batched=True,
                           remove_columns=['input_text','target_text'])
    return train_tok, val_tok


def make_compute_metrics(tokenizer):
    """Returns a compute_metrics function bound to the given tokenizer."""
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        decoded_preds  = [p.strip() for p in decoded_preds]
        decoded_labels = [l.strip() for l in decoded_labels]
        result = rouge_metric.compute(
            predictions=decoded_preds,
            references=decoded_labels,
            use_stemmer=True
        )
        return {
            "rouge1": round(result["rouge1"], 4),
            "rougeL": round(result["rougeL"], 4)
        }
    return compute_metrics


def run_training(model, tokenizer, train_tok, val_tok,
                 experiment_name, learning_rate, num_epochs,
                 batch_size, max_target_len, out_dir):
    """Run Seq2Seq training and return trainer."""
    training_args = Seq2SeqTrainingArguments(
        output_dir=f"{out_dir}/checkpoints_{experiment_name}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        learning_rate=learning_rate,
        weight_decay=0.01,
        fp16=True,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        predict_with_generate=True,
        generation_max_length=max_target_len,
        logging_steps=50,
        report_to="none",
        push_to_hub=False
    )
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=make_compute_metrics(tokenizer),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )
    est = len(train_tok) * num_epochs / (batch_size * 4) / 60
    print(f"Estimated training time: ~{est:.0f} minutes")
    trainer.train()
    return trainer


def save_model(model, tokenizer, experiment_name, out_dir):
    """Save model and tokenizer to Google Drive."""
    path = f'{out_dir}/model_{experiment_name}'
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    print(f"Model saved to: {path}")
    return path


def generate_answers(model, tokenizer, texts,
                     max_input=256, max_target=256,
                     batch_size=16, num_beams=4):
    """Generate predictions in batches."""
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()
    all_answers = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch, return_tensors='pt',
            max_length=max_input, truncation=True, padding=True
        )
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_target,
                num_beams=num_beams,
                early_stopping=True,
                no_repeat_ngram_size=3,
                length_penalty=1.0
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_answers.extend(decoded)
    return all_answers


def save_predictions(test_df, answers, experiment_name, data_dir):
    """Save predictions CSV to Google Drive."""
    out = test_df.copy()
    out['predicted_answer'] = answers
    path = f'{data_dir}/predictions_{experiment_name}.csv'
    out.to_csv(path, index=False)
    print(f"Predictions saved: {path}")
    return path


def plot_learning_curves(trainer, experiment_name, plot_dir):
    """Plot and save training/validation learning curves."""
    log = trainer.state.log_history
    t_steps  = [x['step']      for x in log if 'loss' in x and 'eval_loss' not in x]
    t_losses = [x['loss']      for x in log if 'loss' in x and 'eval_loss' not in x]
    e_epochs = [x['epoch']     for x in log if 'eval_loss' in x]
    e_losses = [x['eval_loss'] for x in log if 'eval_loss' in x]
    e_r1     = [x.get('eval_rouge1', 0) for x in log if 'eval_loss' in x]
    e_rL     = [x.get('eval_rougeL', 0) for x in log if 'eval_loss' in x]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Learning Curves — {experiment_name}', fontsize=13, fontweight='bold')

    axes[0].plot(t_steps, t_losses, color='royalblue', linewidth=1.5)
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.4)

    axes[1].plot(e_epochs, e_losses, color='darkorange', marker='o', linewidth=2)
    axes[1].set_title('Validation Loss per Epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Eval Loss')
    axes[1].grid(True, alpha=0.4)

    axes[2].plot(e_epochs, e_r1, color='green',  marker='o', linewidth=2, label='ROUGE-1')
    axes[2].plot(e_epochs, e_rL, color='purple', marker='s', linewidth=2, label='ROUGE-L')
    axes[2].set_title('ROUGE Scores per Epoch')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('F1 Score')
    axes[2].legend()
    axes[2].grid(True, alpha=0.4)

    plt.tight_layout()
    path = f'{plot_dir}/learning_curves_{experiment_name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {path}")


def save_experiment_log(experiment_name, config, metrics, data_dir):
    """
    Save experiment config and results to a JSON log on Google Drive.
    This lets you resume and track results even if Colab disconnects.
    """
    log_path = f'{data_dir}/experiment_log.json'

    # Load existing log or start fresh
    if os.path.exists(log_path):
        with open(log_path, 'r') as f:
            log = json.load(f)
    else:
        log = {}

    log[experiment_name] = {
        "config":  config,
        "metrics": metrics
    }

    with open(log_path, 'w') as f:
        json.dump(log, f, indent=2)

    print(f"Experiment log updated: {log_path}")
    print(f"Entry saved: {experiment_name}")
    print(json.dumps(log[experiment_name], indent=2))


def check_already_done(experiment_name, data_dir):
    """
    Check if this experiment was already completed.
    Returns True if predictions CSV already exists on Drive.
    Lets you skip re-running experiments after GPU resets.
    """
    pred_path = f'{data_dir}/predictions_{experiment_name}.csv'
    model_path = f'{OUT_DIR}/model_{experiment_name}'
    if os.path.exists(pred_path):
        print(f"ALREADY DONE: {experiment_name}")
        print(f"Predictions found at: {pred_path}")
        print("Skipping training and inference. Load results from Drive.")
        return True
    print(f"NOT DONE YET: {experiment_name} — will run now.")
    return False


print("All helper functions loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


All helper functions loaded.


In [11]:
# CELL S6 — Check experiment log
log_path = f'{DATA_DIR}/experiment_log.json'
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        existing_log = json.load(f)
    print("=== EXPERIMENTS ALREADY COMPLETED ===")
    for exp_name, entry in existing_log.items():
        m = entry.get('metrics', {})
        print(f"  {exp_name}")
        print(f"    val_rouge1={m.get('val_rouge1','?')}, "
              f"val_rougeL={m.get('val_rougeL','?')}, "
              f"zindi_score={m.get('zindi_score','not submitted yet')}")
else:
    print("No experiment log found yet. Starting fresh.")

No experiment log found yet. Starting fresh.


In [ ]:
# Exp01: mt5-small zero-shot baseline
run_experiment(configs[0])

In [ ]:
# Exp03: flan-t5-base LoRA r=16
run_experiment(configs[2])

In [ ]:
# Exp04: flan-t5-base LoRA r=32
run_experiment(configs[3])

In [ ]:
# Exp05: flan-t5-base prompt v2
run_experiment(configs[4])

In [ ]:
# Exp06: flan-t5-large LoRA r=16
run_experiment(configs[5])

In [ ]:
# Run a single experiment by index: 0=exp1, 1=exp2, 2=exp3, 3=exp4, 4=exp5, 5=exp6
run_experiment(configs[1])

Running : exp02_mt5small_finetuned
Model   : google/mt5-small
LoRA    : True (r=16)
LR      : 0.0003
Epochs  : 3
Prompt  : v1
Skip training (zero-shot) : False
Sample input: Language: Aka_Gha
Question: Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye wɔn werɛ, sɛnea wɔbɛboa wɔn ma wɔanya mmoa firi nnwumakuo a ɛfata hɔ, ne sɛnea wɔbɛsiw afɔbu suban ne nsɛm a nkurɔfoɔ de gu oyarefoɔ no so no kwan.
Answer:


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:560: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Total parameters: 300,176,768
trainable params: 688,128 || all params: 300,864,896 || trainable%: 0.2287166130541198


Map:   0%|          | 0/29814 [00:00<?, ? examples/s]

Map:   0%|          | 0/6686 [00:00<?, ? examples/s]

Tokenization complete.
Train tokens: 29814, Val tokens: 6686


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:469: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Estimated training time: ~47 minutes
Starting training...


Epoch,Training Loss,Validation Loss


OverflowError: out of range integral type conversion attempted